# الفصل 28: توسيع العمق: Depth Upscaling

> عكس فصل التقليم تمامًا: بدل تصغير نموذج، ماذا لو أردنا تكبيره دون البدء من الصفر؟

- **الجزء:** 6 — إعادة استخدام النماذج
- **الفصل:** 28
- **المستوى:** متقدم
- **زمن القراءة:** 15 دقيقة
- **المتطلبات:** الفصل 27: التقليم وضغط النماذج: Pruning
- **الوسوم:** Depth Upscaling، SOLAR، Frankenmerging، CPT

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-06-model-reuse/ch28-depth-upscaling.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-06-model-reuse%2Fch28-depth-upscaling.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-06-model-reuse/ch28-depth-upscaling.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-06-model-reuse/ch28-depth-upscaling.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-06-model-reuse/ch28-depth-upscaling.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-06-model-reuse/ch28-depth-upscaling.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** torch
- **GPU:** يستخدم هذا الدفتر PyTorch أو نماذج كبيرة؛ يُفضّل GPU. يمكنك البدء بنماذج صغيرة على CPU، لكن التدريب الكامل سيكون بطيئًا.
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [
    ("torch", "torch")
]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-06-model-reuse/ch28-depth-upscaling.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-06-model-reuse%2Fch28-depth-upscaling.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-06-model-reuse/ch28-depth-upscaling.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-06-model-reuse/ch28-depth-upscaling.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-06-model-reuse/ch28-depth-upscaling.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-06-model-reuse/ch28-depth-upscaling.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> عكس فصل التقليم تمامًا: بدل تصغير نموذج، ماذا لو أردنا تكبيره دون البدء من الصفر؟

> **بيانات الفصل**
>
> - **الجزء:** 6 — إعادة استخدام النماذج
> - **المستوى:** متقدم
> - **زمن القراءة:** 15 دقيقة
> - **المتطلبات:** الفصل 27: التقليم وضغط النماذج: Pruning


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم منهجية SOLAR 10.7B بالتفصيل
- فهم لماذا تنجح هذه الطريقة "العشوائية" ظاهريًا؟
- فهم Depth Upscaling


## 1. لماذا يهمّك هذا الموضوع؟

🟢 طبقة الحدس
عكس فصل التقليم تمامًا: بدل تصغير نموذج، ماذا لو أردنا تكبيره دون البدء من الصفر؟ Depth Upscaling هي التقنية التي استُخدمت لبناء SOLAR 10.7B (Upstage، 2023) — نموذج حقق نتائج متفوقة على نماذج أكبر منه وقت إصداره، عبر استنساخ وتركيب طبقات من نموذج أساسي 7B موجود مسبقًا (Mistral 7B)، ثم إخضاع البنية الموسّعة لتدريب مستمر (CPT) قصير نسبيًا.


## 2. الخريطة الذهنية


```text
طرق تكبير النموذج دون البدء من الصفر
│
├── Depth Upscaling (SOLAR)
│   نسختان من نفس الأساس → تقليم طرفي → تركيب → CPT
│
├── Width Expansion (bert2BERT)
│   توسيع أبعاد الطبقات مع نقل الأوزان بذكاء
│
├── Progressive Stacking
│   إضافة طبقات تدريجيًا أثناء التدريب الأصلي
│
├── Frankenmerging
│   تركيب طبقات من نماذج مختلفة (وليس نسخًا من نفس الأساس)
│
└── From Scratch
    تدريب نموذج بالحجم المستهدف من الصفر (الأغلى)
```


الفرق الجوهري بين Depth Upscaling وFrankenmerging: الأول يبدأ من **نسختين
متطابقتين** من نفس النموذج، فيضمن أن كل طبقة مُدرَّبة ومتوافقة إحصائيًا. الثاني
يخلط طبقات من نماذج مختلفة، وقد ينجح تجريبيًا لكنه بلا ضمانات نظرية.


## 3. الشرح المتدرج

### 3.1 منهجية SOLAR 10.7B بالتفصيل

🟡 طبقة الآلية


```text
خطوات بناء SOLAR 10.7B من Mistral 7B (32 طبقة):

الخطوة 1: أخذ نسختين مطابقتين تمامًا من النموذج الأساسي (32 طبقة لكل منهما)
   نسخة A: طبقات [0, 1, 2, ..., 31]
   نسخة B: طبقات [0, 1, 2, ..., 31]

الخطوة 2: إزالة آخر m طبقة من النسخة A، وأول m طبقة من النسخة B
   (في SOLAR: m=8)
   نسخة A المُعدَّلة: طبقات [0, ..., 23]   (24 طبقة)
   نسخة B المُعدَّلة: طبقات [8, ..., 31]   (24 طبقة)

الخطوة 3: التركيب (Passthrough/Concatenation) المباشر للطبقات
   النموذج الجديد = طبقات A[0..23] + طبقات B[8..31]
   = 24 + 24 = 48 طبقة إجمالاً (من أصل 32+32=64، بإزالة 16 طبقة متداخلة)

الخطوة 4: Continued Pretraining (الفصل 24) على النموذج الموسَّع الجديد
   لـ "لحام" الطبقات المُركَّبة حديثًا معًا بسلاسة، لأن نقطة الوصل
   بين A[23] وB[8] لم تُدرَّب معًا أبدًا من قبل
```


In [ ]:
import torch
import torch.nn as nn
import copy

class DepthUpscaler:
    """
    تطبيق كامل لمنهجية Depth Upscaling بأسلوب SOLAR 10.7B: بناء نموذج
    أعمق من نسختين من نفس النموذج الأساسي عبر التركيب المباشر
    (Passthrough) للطبقات، تمهيدًا لمرحلة Continued Pretraining.
    """
    def __init__(self, base_model, n_layers_to_remove: int = 8):
        self.base_model = base_model
        self.n_layers_to_remove = n_layers_to_remove
        self.original_n_layers = len(base_model.layers)

    def create_upscaled_model(self):
        """
        بناء النموذج الموسَّع: نسختان من النموذج الأساسي، كل منهما
        تُقلَّم من أحد طرفيها، ثم تُركَّبان معًا.
        """
        copy_a = copy.deepcopy(self.base_model)
        copy_b = copy.deepcopy(self.base_model)

        # النسخة A: إزالة آخر n_layers_to_remove طبقة
        layers_a = copy_a.layers[: self.original_n_layers - self.n_layers_to_remove]

        # النسخة B: إزالة أول n_layers_to_remove طبقة
        layers_b = copy_b.layers[self.n_layers_to_remove:]

        # التركيب المباشر: كل طبقات A يليها كل طبقات B
        upscaled_layers = nn.ModuleList(list(layers_a) + list(layers_b))

        upscaled_model = copy.deepcopy(self.base_model)
        upscaled_model.layers = upscaled_layers

        new_n_layers = len(upscaled_layers)
        print(f"عدد الطبقات: الأصلي={self.original_n_layers} → "
              f"الموسَّع={new_n_layers} "
              f"(تضمين {self.original_n_layers * 2 - new_n_layers} "
              f"طبقة متداخلة مُزالة)")

        return upscaled_model

    def validate_layer_continuity(self, upscaled_model, sample_input):
        """
        فحص سريع قبل CPT: التأكد أن إخراج النموذج الموسَّع "معقول"
        إحصائيًا (ليس NaN أو قيمًا متفجرة)، حتى قبل أي تدريب إضافي -
        هذا ممكن لأن الطبقات المُركَّبة كل منها مُدرَّبة أصلاً جيدًا
        على مدخلات ومخرجات من نفس "النمط" العام، فالنموذج الموسَّع
        غير المُدرَّب لا يزال "يعمل" بشكل خام معقول، وإن لم يكن أمثل.
        """
        with torch.no_grad():
            output = upscaled_model(sample_input)
            has_nan = torch.isnan(output.logits).any().item()
            output_std = output.logits.std().item()

        return {
            "has_nan": has_nan,
            "output_std": output_std,
            "status": "✅ جاهز لـ CPT" if not has_nan else "❌ مشكلة - راجع نقطة الدمج",
        }


### 3.2 لماذا تنجح هذه الطريقة "العشوائية" ظاهريًا؟

🔴 طبقة التعمّق

قد يبدو تركيب نصفي نموذجين معًا دون أي تدريب "عشوائيًا" وغير منطقي، لكن هناك تفسيرين رئيسيين لنجاحها:


```text
1. التشابه البنيوي للطبقات المتوسطة في Transformer
   أبحاث التفسير (Mechanistic Interpretability - الفصل 61) تُظهر أن
   الطبقات المتوسطة في معظم نماذج Transformer تُنتج تمثيلات (hidden
   states) ذات توزيع إحصائي متشابه نسبيًا (مقارنة بالطبقة الأولى
   والأخيرة المتخصصتين جدًا)، لذا فإن "قطع ولصق" طبقات من نفس
   النموذج الأساسي عند نقطة متوسطة لا يُسبِّب انقطاعًا كارثيًا.

2. Residual Stream كـ"ممر آمن"
   بنية Transformer القائمة على Residual Connections (الفصل 11) تعني
   أن كل طبقة "تُضيف" تحديثًا صغيرًا نسبيًا لتيار رئيسي مستمر
   (Residual Stream)، بدل تحويل التمثيل بالكامل. حتى لو كانت نقطة
   الوصل بين النصفين غير مثالية، فإن بنية الـ Residual تمنع انهيار
   المعلومة الأساسية المارة عبر الشبكة، مما يجعل "اللحام" عبر
   تدريب CPT قصير نسبيًا كافيًا لتصحيح أي خلل محلي عند نقطة الوصل.
```


### 3.3 مقارنة Depth Upscaling بتقنيات نمو النموذج الأخرى


## 4. مثال محلول خطوة بخطوة


In [ ]:
def estimate_upscaling_cost_savings(
    base_model_params_billions: float,
    target_model_params_billions: float,
    cpt_tokens_billions: float,
    from_scratch_tokens_billions: float,  # وفق Chinchilla (الفصل 20): ≈20×N
) -> dict:
    """
    مقارنة تقديرية للتكلفة الحاسوبية (FLOPs) بين بناء نموذج بالحجم
    المستهدف عبر Depth Upscaling + CPT، مقابل تدريبه من الصفر بالكامل.
    """
    # تقدير FLOPs بقانون Kaplan/Chinchilla المبسّط: FLOPs ≈ 6 × N × D
    flops_upscaling_cpt = 6 * (target_model_params_billions * 1e9) * (cpt_tokens_billions * 1e9)
    flops_from_scratch = 6 * (target_model_params_billions * 1e9) * (from_scratch_tokens_billions * 1e9)

    savings_factor = flops_from_scratch / flops_upscaling_cpt

    return {
        "flops_upscaling_approach": f"{flops_upscaling_cpt:.2e}",
        "flops_from_scratch_approach": f"{flops_from_scratch:.2e}",
        "savings_factor": f"{savings_factor:.1f}×",
        "conclusion": (
            f"بناء نموذج {target_model_params_billions}B عبر Depth Upscaling "
            f"من نموذج {base_model_params_billions}B موجود يوفر تقريبًا "
            f"{savings_factor:.0f} ضعف الحوسبة مقارنة بالتدريب من الصفر"
        ),
    }

result = estimate_upscaling_cost_savings(
    base_model_params_billions=7.0,
    target_model_params_billions=10.7,
    cpt_tokens_billions=3,       # SOLAR استخدمت تدريبًا إضافيًا محدودًا نسبيًا
    from_scratch_tokens_billions=10.7 * 20,  # وفق Chinchilla لنموذج 10.7B
)
print(result)
# النتيجة التقريبية: savings_factor ≈ 71× - توفير هائل جدًا في الحوسبة!


## 5. نصائح وحيل

- **اختر `m` بنسبة 20–30% من عمق الأساس** كنقطة بداية (SOLAR استخدمت 8 من 32).
- **تحقّق من استقرار المخرجات قبل CPT** (لا NaN، وstd معقول) لتوفير تكلفة تدريب فاشل.
- **استخدم CPT قصيرًا مع Replay** من البيانات العامة لتقليل النسيان الكارثي.
- **اخفض معدل التعلّم** في بداية CPT (Re-warming) ثم استخدم جدولة cosine.
- **قيّم عند نقطة الوصل** (بين آخر طبقة من A وأول طبقة من B) بمهام تشخيصية، لا بالخسارة وحدها.
- **جرّب أكثر من `m`** على نطاق صغير قبل الالتزام بتدريب كامل.
- **حافظ على نفس التوكنizer**؛ تغييره يبطل فكرة إعادة استخدام الطبقات.
- **لا تتوقع نموذجًا جاهزًا بعد التركيب**؛ الطبقات الجديدة لم تُدرَّب معًا بعد.


## 6. أخطاء شائعة ومصائد

اختيار n_layers_to_remove بشكل عشوائي دون تجريب — القيمة المثلى تعتمد على عمق النموذج الأساسي الكلي؛ SOLAR استخدمت 8 من أصل 32 (25%) كنقطة توازن مُختبَرة تجريبيًا.
تخطي مرحلة Continued Pretraining بعد التركيب معتقدين أن النموذج "جاهز فورًا" — بدون CPT، الأداء عند نقطة الوصل سيكون ضعيفًا جدًا رغم أن النموذج "لا ينهار" تمامًا كما أوضحنا.
تطبيق Depth Upscaling على نموذجين مختلفين تمامًا (ليس نسختين من نفس الأساس) متوقعين نفس النجاح — هذا أقرب لـ Frankenmerging (الفصل 26) الذي يتطلب توافقًا بنيويًا ومعرفيًا أعمق بكثير بين مصدري الطبقات.


## 7. جدول مقارنة

| التقنية | الفكرة الأساسية | تكلفة التدريب اللاحق | مثال/مصدر |
|---|---|---|---|
| Depth Upscaling (SOLAR) | تركيب نسختين مُقلَّمتين من نفس النموذج | CPT متوسط (مليارات رموز قليلة) | SOLAR 10.7B |
| Progressive Stacking | إضافة طبقات جديدة تدريجيًا أثناء التدريب الأصلي من الصفر | يبدأ من الصفر أصلًا - ليس إعادة استخدام | bert2BERT، StackBERT |
| Frankenmerging (الفصل 26) | تركيب طبقات من نماذج مختلفة تمامًا (ليس نسخًا من نفس الأساس) | يعتمد على مدى توافق النماذج - أحيانًا بلا تدريب إضافي | Goliath-120B |
| Width Expansion (bert2BERT) | توسيع عرض الطبقات (أبعاد أكبر) بدل عمقها، مع نقل الأوزان بذكاء | تدريب إضافي متوسط | bert2BERT |


## 8. ملخص أفضل الممارسات

✅ استخدم Depth Upscaling عندما يكون هدفك زيادة السعة الإجمالية (Capacity) للنموذج دون تكلفة تدريب من الصفر، خصوصًا عند توفر نموذج أساسي قوي بالفعل بنفس المعمارية.
✅ خصِّص دائمًا مرحلة CPT بعد التركيب — لا تتوقع أداءً جيدًا من النموذج الموسَّع "الخام" مباشرة.
✅ جرّب عدة قيم لـ n_layers_to_remove على نطاق صغير قبل الالتزام بالتدريب الكامل.
✅ تحقق دومًا من استقرار المخرجات (validate_layer_continuity) قبل الاستثمار في تدريب CPT مكلف.


## 9. أسئلة مقابلات

**سؤال: اشرح لماذا يُعتبر Depth Upscaling فعّالًا من حيث التكلفة مقارنة بالتدريب من الصفر، رغم أن النموذج الناتج يحتاج تدريبًا إضافيًا (CPT) لاحقًا؟**

**إجابة نموذجية:** بناء نموذج بحجم N معامل من الصفر يتطلب وفق قوانين Chinchilla حوالي 20×N رمز تدريبي للوصول لأداء مُحسَّن حسابيًا (Compute-Optimal) — لنموذج 10.7B هذا يعني أكثر من 200 مليار رمز. أما Depth Upscaling فيبدأ من نموذج 7B مُدرَّب بالفعل (استهلك هو نفسه مليارات الرموز في تدريبه الأصلي، لكن هذه التكلفة "غارقة" بالفعل ومدفوعة)، ثم يُركِّب طبقات منه لإنتاج نموذج 10.7B يحتاج فقط لبضعة مليارات رموز إضافية من CPT (وليس 200+ مليار) لـ"لحام" نقاط الوصل الجديدة وتحسين الأداء العام. السبب الجوهري في فعالية هذا: الطبقات المُركَّبة ليست عشوائية التهيئة (كما في التدريب من الصفر) بل تحمل بالفعل معرفة لغوية وعالمية غنية مُكتسَبة من تدريبها الأصلي، فمهمة CPT هنا هي "ضبط دقيق لتوافق البنية الجديدة" وليس "تعليم من الصفر" — وهذا فارق جوهري في حجم البيانات والحوسبة المطلوبة.


## 10. تمرين عملي

باستخدام DepthUpscaler، وسِّع نموذج GPT-2 الصغير (12 طبقة) إلى نموذج بـ 18 طبقة (بإزالة 6 طبقات من كل نسخة)، وقارن Perplexity النموذج الموسَّع الخام (قبل أي CPT) مقابل النموذج الأصلي 12 طبقة على نص اختباري، ثم طبّق CPT قصيرًا (بضع مئات من الخطوات) وأعد القياس لتوثيق التحسّن.


## 11. مراجع للتعمّق

- Kim et al. (2023, Upstage): SOLAR 10.7B: Scaling Large Language Models with Simple yet Effective Depth Up-Scaling
- Chen et al. (2021): bert2BERT: Towards Reusable Pretrained Language Models
- Gong et al. (2019): Efficient Training of BERT by Progressively Stacking


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — إعادة استخدام النماذج / الفصل 28: توسيع العمق: Depth Upscaling.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
